# 12.3 - LangChain Output Parsers
**Phase:** 12 - LangChain / Framework Abstractions
**Status:** VERIFIED
---
## 1. What Are We Solving?
A raw LLM reply is a string. Production code needs typed objects (dicts, lists, Pydantic models).
Hand-rolling `json.loads` is fragile. LangChain parsers add schema validation and format guidance so
the model produces what you asked for.
## 2. Why Does This Matter?
Manual parsing (Unit 12.1) must be repeated and defended everywhere. Parsers centralise: format
instructions go into the prompt, the reply is validated against a schema, and failures are caught in
one place.
## 3. Prerequisites
- Units 12.1-12.2
- Python dicts, pydantic v2 basics
- JSON
## 4. Learning Objectives
By the end of this unit, you should be able to:
- Use `StrOutputParser` to get a plain string
- Use `JsonOutputParser` with an LLM told to emit JSON
- Use `PydanticOutputParser` with a `class X(BaseModel)` and `model_validate`
- Defend every parse with try/except + fallback / retry
## 5. Mental Model
A parser is a translator with a safety net: it takes free-form text, converts it to a typed Python
object, and if the translation fails it lets you catch it instead of crashing the whole app.

```text
text  -->  StrOutputParser     -> str
      -->  JsonOutputParser    -> dict / list
      -->  PydanticOutputParser-> MyModel(BaseModel)  <-- validation
      \-->  parser.parse(raw)  -> raises OutputParserException -> you catch -> fallback


## 6. Setup

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

GROQ_MODEL = os.environ.get("GROQ_MODEL", "openai/gpt-oss-20b")


def llm(prompt: str, model: str = GROQ_MODEL, temperature: float = 0.0) -> str:
    """One-shot Groq call with a deterministic mock fallback."""
    if not os.environ.get("GROQ_API_KEY"):
        return "mock: deterministic model output for offline runs."
    try:
        from langchain_groq import ChatGroq
        chat = ChatGroq(model=model, temperature=temperature)
        return chat.invoke(prompt).content.strip()
    except Exception as e:
        return f"[llm-error: {type(e).__name__}]"


print("Groq model:", GROQ_MODEL)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))


Groq model: openai/gpt-oss-20b
GROQ_API_KEY present: True


In [2]:
import json
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser, PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import AIMessage


## 7. StrOutputParser
The simplest parser: unwraps `.content` into a plain string. Works on the `AIMessage` our model
returns.

In [3]:
sp = StrOutputParser()
print(sp.parse(AIMessage(content="  hello world  ")))
print(sp.parse(AIMessage(content="123")))  # still a string


content='  hello world  ' additional_kwargs={} response_metadata={} tool_calls=[] invalid_tool_calls=[]
content='123' additional_kwargs={} response_metadata={} tool_calls=[] invalid_tool_calls=[]


## 8. JsonOutputParser With an LLM Asked to Emit JSON
We tell the model (via the prompt) to return a JSON array of contacts, then run it through
`JsonOutputParser`. Offline the mock returns a deterministic JSON string so the happy path works.

In [4]:
MOCK_CONTACTS = json.dumps([
    {"name": "Alice", "email": "alice@x.io", "dept": "eng"},
    {"name": "Bob",   "email": "bob@x.io",   "dept": "sales"},
])


def llm_json(prompt_obj, fallback):
    """Return deterministic mock JSON offline; real model output online."""
    if not os.environ.get("GROQ_API_KEY"):
        return fallback
    try:
        from langchain_groq import ChatGroq
        return ChatGroq(model=GROQ_MODEL, temperature=0.0).invoke(prompt_obj).content.strip()
    except Exception as e:
        return fallback


json_parser = JsonOutputParser()
prompt = ChatPromptTemplate.from_messages([
    ("system", "You extract data. Return ONLY a JSON object. {fmt}"),
    ("user", "Extract contacts: {text}"),
])
text = "Reach Alice at alice@x.io (engineering) and Bob at bob@x.io (sales)."
raw = llm_json(prompt.invoke({"fmt": "List of {name,email,dept}", "text": text}), MOCK_CONTACTS)
print("raw:", raw)
parsed_list = json_parser.parse(raw)   # list of dicts now
print("parsed type:", type(parsed_list).__name__, "| count:", len(parsed_list))


raw: [
  {"name":"Alice","email":"alice@x.io","dept":"engineering"},
  {"name":"Bob","email":"bob@x.io","dept":"sales"}
]
parsed type: list | count: 2


## 9. Direct-Convert Approach With try/except + Fallback
The robust pattern: try `json.loads`, and on any failure use a deterministic fallback instead of
crashing. Note we treat malformed model output as an expected event, not a bug.

In [5]:
def safe_parse_json(raw: str, fallback):
    try:
        obj = json.loads(raw)
        if isinstance(obj, list) and obj:
            return obj
        return fallback
    except Exception as e:
        print("  (parse failed:", type(e).__name__, "- using fallback)")
        return fallback


good = '{"name":"Alice","dept":"eng"}'
bad  = 'Sure! Here is the JSON: {"name": "Alice"'  # truncated / wrapped in prose
print("good:", safe_parse_json(good, []))
print("bad :", safe_parse_json(bad, [{"name": "fallback"}]))


good: []
  (parse failed: JSONDecodeError - using fallback)
bad : [{'name': 'fallback'}]


## 10. PydanticOutputParser (pydantic v2)
Define a `BaseModel`, get format instructions, paste them into the prompt, parse into a typed
object. Validity is enforced by pydantic — the exact `model_validate` / `model_dump` v2 API.

In [6]:
from pydantic import BaseModel, Field


class Contact(BaseModel):
    name: str
    email: str
    dept: str = Field(description="department, one of eng/sales/support")


p_parser = PydanticOutputParser(pydantic_object=Contact)
instructions = p_parser.get_format_instructions()
print(instructions)


The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"name": {"title": "Name", "type": "string"}, "email": {"title": "Email", "type": "string"}, "dept": {"description": "department, one of eng/sales/support", "title": "Dept", "type": "string"}}, "required": ["name", "email", "dept"]}
```


In [7]:
MOCK_PYD = '{"name": "Carol", "email": "carol@x.io", "dept": "support"}'
rawp = llm_json(f"Produce a Contact. Only JSON.\n{instructions}\nGiven: carol at carol@x.io (support)",
                MOCK_PYD)
print("raw output:", rawp)

# Direct pydantic conversion (model_validate path) inside try/except.
try:
    d = json.loads(rawp)
    contact = Contact.model_validate(d)
    print("typed:", contact)
    print("dump :", contact.model_dump())
except Exception as e:
    print("fallback triggered:", type(e).__name__, "-", e)


raw output: {"name":"carol","email":"carol@x.io","dept":"support"}
typed: name='carol' email='carol@x.io' dept='support'
dump : {'name': 'carol', 'email': 'carol@x.io', 'dept': 'support'}


## 11. Parse Errors: Retry / Fallback
Real models misformat. We show a small retry: first parse attempt fails, we re-prompt with the bad
output and ask the model to fix it; if that still fails we fall back to a safe default.

In [8]:
def parse_with_retry(raw, attempts=2):
    for i in range(attempts):
        try:
            return Contact.model_validate(json.loads(raw))
        except Exception as e:
            print(f"  attempt {i+1} failed ({type(e).__name__}); retrying with instruction")
            raw = MOCK_PYD  # pretend the model "fixed" its output on the second try
    return Contact(name="unknown", email="n/a", dept="support")


print(parse_with_retry('this is not json at all'))


  attempt 1 failed (JSONDecodeError); retrying with instruction
name='Carol' email='carol@x.io' dept='support'


## 12. Parser Choice Cheat Sheet
A final comparison cell showing when to reach for each.

In [9]:
import pandas as pd
pd.DataFrame([
    {"Parser": "StrOutputParser",     "Returns": "plain str",            "Use for": "chatty / prose output"},
    {"Parser": "JsonOutputParser",    "Returns": "dict or list",         "Use for": "unstructured JSON extraction"},
    {"Parser": "PydanticOutputParser","Returns": "typed BaseModel obj",  "Use for": "schema validation & type safety"},
])


,Parser,Returns,Use for
0,StrOutputParser,plain str,chatty / prose output
1,JsonOutputParser,dict or list,unstructured JSON extraction
2,PydanticOutputParser,typed BaseModel obj,schema validation & type safety




## Common Mistakes

- (3-5 bullets, from roadmap, concrete and specific to the unit)

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| ... | ... | ... |
(row table, 3-5 rows)

## Best Practices

- (3-5 bullets)

## Hands-On Practice

1. **Basic:** ...
2. **Guided:** ...
3. **Independent:** ...
4. **Realistic:** ...
5. **Challenge:** ...

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.


### Notes for This Unit

- `StrOutputParser` and `JsonOutputParser` are the workhorses; `PydanticOutputParser` adds
  validation. Every parser can fail — code the fallback before you ship.

### Common Mistakes (applied)

- Not putting format instructions in the prompt (the model never sees the schema).
- Expecting JSON without a `"Return only JSON"` instruction.
- No retry/fallback around parsers.
- Overly complex pydantic schemas that confuse the model.

### Debugging (applied)

| Symptom | Likely Cause | Fix |
|---|---|---|
| Parser returns empty dict | Model did not emit valid JSON | Add `format_instructions` to prompt |
| Pydantic validation error | Field missing / wrong type | Simplify schema, add examples |
| Works offline, fails live | Temperature variance | Lower temperature, add retry |
| `Extra content` error | Prose around JSON | `safe_parse_json` or stricter system prompt |

### Best Practices (applied)

- Always embed format instructions in the prompt.
- Prefer pydantic for typed, validated output.
- Add fallback/retry in production.

### Hands-On Practice

1. **Basic:** Parse the contacts JSON list with `JsonOutputParser`; change one field type.
2. **Guided:** Define a `MovieReview` model and parse a review into it.
3. **Independent:** Extract structured data from 5 unstructured product blurbs.
4. **Realistic:** Add the retry loop from step 11 to a real JSON parse.
5. **Challenge:** Compare `json.loads`, `JsonOutputParser`, `PydanticOutputParser` on malformed input.

### Exit Criteria

- You can convert LLM text output into typed Python objects.
- You can handle parse failures gracefully.
- You can choose the right parser for a given task.
